# Домашняя работа №3: Построение продвинутого пайплайна классификации для свёрточных сетей

В этой домашней работе мы продолжим работать с датасетом Tiny ImageNet и постараемся сильно улучшить предыдущий результат посредством построения более продвинутого пайплайна обучения без изменения самой модели. В рамках этого задания желательно продолжить с той же архитектурой, которую вы использовали в предыдущем домашнем задании, — так вы напрямую увидите, насколько сильное влияние оказывает сам тренировочный процесс и что не всегда стоит бежать менять архитектуру, столкнувшись с неудовлетворительным качеством работы сети :) Итак, приступим!

## Часть 0: Подготовка

Импортируем необходимые библиотеки.

In [ ]:
import io
from time import time
from typing import Union, Callable, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.v2 as transforms
import pandas as pd
import numpy as np
import PIL.Image as Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary

Скопируем пайплайн тренировки из предыдущего домашнего задания.

In [ ]:
def run_epoch(model: nn.Module, loader: DataLoader, criterion: Callable, optimizer: Optional[torch.optim.Optimizer] = None,\
              scheduler: Optional[torch.optim.lr_scheduler.LRScheduler] = None, device: torch.device = torch.device("cpu")) -> torch.Tensor:
    all_labels, all_preds = [], []
    loss_epoch = 0.
    for batch in loader:
        images, labels = batch
        images, labels = images.to(device), labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        if optimizer is not None:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if scheduler is not None:
                scheduler.step()

        loss_epoch += loss.item()
        preds = torch.argmax(logits.softmax(dim=-1), dim=-1)
        if len(labels.size()) > 1:
            labels = labels.argmax(dim=-1)

        all_preds = np.concatenate((all_preds, preds.cpu().numpy()))
        all_labels = np.concatenate((all_labels, labels.cpu().numpy()))

    loss_epoch /= len(loader)
    acc_epoch = (all_preds == all_labels).sum() / len(all_preds)

    return loss_epoch, acc_epoch

def train(model: nn.Module, n_epochs: int, train_loader: DataLoader, criterion: Callable, optimizer: torch.optim.Optimizer,
          scheduler: Optional[torch.optim.lr_scheduler.LRScheduler] = None, val_loader: Optional[DataLoader] = None, val_freq: int = 10,\
          save_best: bool = True, save_name: str = 'model', device: torch.device = torch.device("cpu")) -> nn.Module:
    enable_validation = val_loader is not None
    best_val = 0.

    for epoch in range(n_epochs):
        timer_start = time()
        model.train()
        train_loss_epoch, train_acc_epoch = run_epoch(model, train_loader, criterion, optimizer, scheduler, device)

        print(f"Epoch {epoch+1}:")
        print(f"Train loss: {train_loss_epoch} | Train acc: {train_acc_epoch * 100}%")

        if enable_validation and epoch % val_freq == 0:
            model.eval()
            with torch.no_grad():
                val_loss_epoch, val_acc_epoch = run_epoch(model, val_loader, criterion, optimizer=None, scheduler=None, device=device)

            if save_best and val_acc_epoch >= best_val:
                best_val = val_acc_epoch
                model.to("cpu")
                torch.save(model.state_dict(), f"{save_name}.pth")
                model.to(device)

            print(f"Val loss: {val_loss_epoch} | Val acc: {val_acc_epoch * 100}%")

        print(f"Time spent on epoch: {time() - timer_start}")

    return model

Скопируйте класс датасета из предыдущего задания.

In [ ]:
class TinyImageNetDataset(Dataset):
    ...

Скопируйте функцию `stratified_train_val_split` из предыдущего задания.

In [ ]:
def stratified_train_val_split(df: pd.DataFrame, train_share: float, seed: int = 42) -> tuple[pd.DataFrame, pd.DataFrame]:
    pass

Скопируйте вашу архитектуру модели и необходимые блоки.

In [ ]:
class MobileNetV3(nn.Module):
    ...

Загрузим данные и разобьём их на train-/val-части.

In [ ]:
data_path = r"C:\Users\asang\Documents\Study\Karpov\Lesson 3\data\train-00000-of-00001-1359597a978bc4fa.parquet" # замените на путь до .parquet файла с train частью датасета
df = pd.read_parquet(data_path, engine='fastparquet')
df.drop(columns=['image.path'], inplace=True)

train_df, val_df = stratified_train_val_split(df, train_share=0.9, seed=42)

## Часть 1: Пайплайн аугментации

Постройте пайплайн аугментации, примените подходы, разобранные на лекциях и практике, попробуйте свои идеи. Список аугментаций, доступный в torchvision, приведён тут: https://pytorch.org/vision/stable/transforms.html. В качестве одного из элементов пайплайна рекомендуем обратить внимание на RandAugment:

**RandAugment** — это алгоритм автоматического аугментирования изображений, разработанный Google Research. Его основная идея заключается в случайном применении набора простых операций преобразования изображений (таких, как поворот, изменение яркости, контраста, обрезка и т.д.). Оригинальная статья: https://arxiv.org/pdf/1909.13719v2.

Он довольно прост в настройке, т. к. принимает всего два гиперпараметра: *N* — количество аугментаций, применяемых за раз, *M* — сила каждой аугментации. Полный список аугментаций:
- RandomShear
- RandomTranslation
- RandomRotation
- RandomBrightness
- RandomColor
- RandomContrast
- Posterize
- Solarize
- Equalize
- AutoContrast

In [ ]:
train_transform = transforms.Compose([
    ...
])

val_transform = transforms.Compose([
    transforms.PILToTensor(),
    transforms.ToDtype(dtype=torch.float32, scale=True)
])

Рассмотрим также более продвинутые способы аугментации — **MixUp** и **CutMix**. Традиционные методы аугментации (например, поворот, масштабирование, изменение яркости) работают с одним изображением, применяя к нему различные преобразования. MixUp и CutMix же работают сразу с парами изображений и, как можно судить из названий, каким-то образом смешивают их.

Начнём с MixUp. Он действует следующим образом: берутся два изображения и их метки, затем создаётся новое изображение путём их линейного смешивания. Представьте, что у вас есть фотография кошки и фотография собаки. MixUp накладывает их друг на друга с определённым коэффициентом $\lambda$ (например, 0.6 кошка + 0.4 собака). При этом метка нового изображения также становится смешанной: [0.6, 0.4].

CutMix работает иначе: вместо смешивания всего изображения он вырезает прямоугольную область из одного изображения и вставляет её в другое. Метки также смешиваются, но пропорционально площади вырезанной области. Если мы вырезали 30% площади из изображения собаки и вставили в изображение кошки, метка будет [0.7, 0.3].

Дополнительное отличие этих двух методов от тех, которые мы использовали в первой части, — способ встраивания в пайплайн обучения. Традиционные аугментации обычно применяются на этапе предобработки данных перед началом формирования батча. MixUp и CutMix в свою очередь же работают с батчами, а не с индивидуальными картинками, поэтому требуют иной логики встраивания в пайплайн: самый простой способ — это применять эти аугментации непосредственно в тренировочном цикле, но в таком случае мы никак не пользуемся мультипроцессингом нашего дата-лоадера. Более оптимальный подход — передавать в data loader модифицированную collate_fn, где на батч применяются наши аугментации, поэтому напишем `collate_fn` для применения этих аугментаций. Документация по `collate_fn`: https://pytorch.org/docs/stable/data.html#working-with-collate-fn. По сути, когда включено автоматическое формирование батчей (дефолт), эта функция принимает на вход лист из семплов, полученных вызовом метода `__getitem__` у нашего датасета, а потом формирует из него батч. Стандартная `torch.utils.data.default_collate` просто стекает их по батч-дименшену и конвертит в `torch.Tensor`.

In [ ]:
... # определите здесь CutMix/MixUp

def collate_fn(batch):
    return ...

Отлично, теперь попробуйте построить собственный пайплайн аугментаций.

Примечание: аугментации выше даны для примера, необязательно использовать их, возможно, вы соберёте оптимальный набор из совершенно других методов.

Примечание 2: если будете использовать CutMix/MixUp, то рекомендуем увеличить количество эпох, поскольку они дают достаточно сильную регуляризацию.

In [ ]:
train_dataset, val_dataset = TinyImageNetDataset(train_df, train_transform), TinyImageNetDataset(val_df, val_transform)

# Если используете MixUp/CutMix, не забудьте добавить в train_loader collate_fn=collate_fn
train_loader, val_loader = DataLoader(train_dataset, batch_size=200, shuffle=True, pin_memory=True, drop_last=False),\
                           DataLoader(val_dataset, batch_size=200, pin_memory=True, shuffle=False)

n_epochs = 30
criterion = nn.CrossEntropyLoss()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = MobileNetV3(3, 200)
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, foreach=False, fused=True)

model = train(model, n_epochs, train_loader, criterion, optimizer, val_loader, val_freq=1, save_best=True, save_name="model_p1" device=device)

Отлично! Можем увидеть, что даже добавление простых аугментаций значительно снижает переобучение модели и повышает итоговый скор. Загрузите код модели и веса в LMS для проверки в приватном тесте.

## Часть 2: Настройка процесса оптимизации модели

Теперь тренировочный датасет стал намного более вариативным, настало время перейти к настройке процесса оптимизации модели. В этой части ДЗ предлагаем вам подобрать подходящий оптимизатор, scheduler (если понадобится) и Learning Rate.

Скопируйте пайплайны аугментации, которые вы получили в прошлой части задания.

In [ ]:
train_transform = transforms.Compose([
    ...
])

val_transform = transforms.Compose([
    transforms.PILToTensor(),
    transforms.ToDtype(dtype=torch.float32, scale=True)
])

In [ ]:
train_dataset, val_dataset = TinyImageNetDataset(train_df, train_transform), TinyImageNetDataset(val_df, val_transform)

train_loader, val_loader = DataLoader(train_dataset, batch_size=200, shuffle=True, pin_memory=True, drop_last=False),\
                           DataLoader(val_dataset, batch_size=200, pin_memory=True, shuffle=False)

n_epochs = 30
criterion = nn.CrossEntropyLoss()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = MobileNetV3(3, 200)
model.to(device)

optimizer = ...
lr_scheduler = ...

model = train(model, n_epochs, train_loader, criterion, optimizer, lr_scheduler, val_loader, val_freq=1, save_name='model_p2_1cycle2e3', device=device)

Отправьте код модели и натренированный вес в LMS.

## Часть 3: Focal Loss

В следующих 3 частях начнём адресовать проблему шумных лейблов через лосс-функции. В этой части предлагаем реализовать Focal Loss, про который уже рассказывали на лекции. Обычно он используется для датасетов с сильным дисбалансом классов, но его также можно применять и для кейса с шумными лейблами, поскольку он снижает их влияние на тренировочный процесс, если выставить более низкое значение гиперпараметра $\gamma$. Напомним формулу лосса:
$$ \text{FL}(p_{t,c}) = -\sum_{c=1}^C \alpha_c(1-p_{t,c})^\gamma y_c\log(p_{t,c}). $$

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha: Union[float, list, tuple] = 1., gamma: float = 2., reduction: str = 'mean'):
        super().__init__()

        assert reduction in ['none', 'mean', 'sum'], f"{reduction} should be one of {['none', 'mean', 'sum']}"

        self.alpha = torch.tensor(alpha) if isinstance(alpha, (list, tuple)) else alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ...

        focal_loss = ...

        if self.reduction == 'none':
            return focal_loss
        elif self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()

In [ ]:
test_logits = torch.tensor([[1.27, 0.15, 0.03], [0.37, 0.15, 0.33], [0.16, 0.09, 1.73]])
test_labels = torch.tensor([0, 2, 2])

loss = FocalLoss()
assert torch.isclose(loss(test_logits, test_labels), torch.tensor(0.1823), rtol=1e-3), "Incorrect value"

In [ ]:
Отлично! Отправьте код в LMS на проверку.

## Часть 4: Generalized Cross Entropy Loss

Функция GCE была предложена как обобщение стандартной кросс-энтропии, которое делает модель более устойчивой к шуму в данных и неправильным меткам. Формула:
$$ L_{GCE}(p, y) = \frac{(1 - p_y^q)}{q}, $$
где $p_y$ — предсказанная вероятность принадлежности к классу $y$, $q$ — гиперпараметр, который варьируется от 0 до 1. Некоторые свойства этой лосс-функции:
1. Когда $q \to 1$, функция приближается к MAE, т. е. $\lim_{q \to 1} L_{GCE} = 1 - p_y$.
2. Когда $q \to 0$, функция приближается к обычной CE, т. е. $\lim_{q \to 0} L_{GCE} = -\log(p_y)$ (правило Лопиталя).

Идея в том, что, контролируя гиперпараметр $q$, мы можем изменять чувствительность лосса к шуму, т. к. MAE более устойчива к выбросам. Ссылка на оригинальную статью: https://arxiv.org/abs/1805.07836. Предлагаем имплементировать GCE ниже.

In [ ]:
class GeneralizedCrossEntropy(nn.Module):
    def __init__(self, q: float = 0.7, reduction: str = 'mean'):
        super().__init__()

        assert q <= 1.0 and q > 0., "Incorrect q value"
        assert reduction in ['none', 'mean', 'sum'], f"{reduction} should be one of {['none', 'mean', 'sum']}"

        self.q = q
        self.reduction = reduction

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ...

        if self.reduction == 'none':
            return loss
        elif self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()

In [ ]:
test_logits = torch.tensor([[1.27, 0.15, 0.03], [0.37, 0.15, 0.33], [0.16, 0.09, 1.73]])
test_labels = torch.tensor([0, 2, 2])

loss = GeneralizedCrossEntropy(q = 0.7)
assert torch.isclose(loss(test_logits, test_labels), torch.tensor(0.4850), rtol=1e-3), "Incorrect value"

Загрузите код в LMS на проверку.

## Часть 5: Подбор лосс-функции

Предлагаем вам опробовать реализованные выше лосс-функции, а также Label Smoothing в деле. Скопируйте пайплайн обучения из части 2 и подберите лучшую лосс-функцию.

Примечание: оптимальная настройка лосс-функции — весьма сложная задача, поэтому в данной части от вас ожидается, скорее, валидация работоспособности реализованных вами лосс-функций из частей 3 и 4, а также проверка подхода с Label Smoothing в деле. Если не получится добиться улучшения, то решение с обычной CE должно тоже подойти.

In [ ]:
train_transform = transforms.Compose([
    ...
])

val_transform = transforms.Compose([
    transforms.PILToTensor(),
    transforms.ToDtype(dtype=torch.float32, scale=True)
])

train_dataset, val_dataset = TinyImageNetDataset(train_df, train_transform), TinyImageNetDataset(val_df, val_transform)

train_loader, val_loader = DataLoader(train_dataset, batch_size=200, shuffle=True, pin_memory=True, drop_last=False),\
                           DataLoader(val_dataset, batch_size=200, pin_memory=True, shuffle=False)

n_epochs = 30

criterion = ...

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = MobileNetV3(3, 200)
model.to(device)

optimizer = ...
lr_scheduler = ...

model = train(model, n_epochs, train_loader, criterion, optimizer, lr_scheduler, val_loader, val_freq=1, save_name='model_p3_ls', device=device)

Загрузите код модели и её веса в LMS.

## Часть 6: Ансамблирование модели

Рассмотрим ещё одну технику по улучшению качества модели — построение ансамбля модели из её весов в предыдущих эпохах. Два основных способа: Stochastic Weight Averaging (SWA) и Exponential Moving Average (EMA).

### Exponential Moving Average

Метод фактически предлагает вместо одного набора весов n-й эпохи брать их сглаженные значения, которые также учитывают наборы весов предыдущих эпох, т. е.
$$ \theta_{EMA} = \beta \theta_{EMA_{prev}} + (1-\beta) \theta_{current}, $$
где $\theta_{EMA}$ — сглаженный набор весов, $\theta_{current}$ — набор весов в текущей эпохе, $\beta$ — гиперпараметр, отвечающий за «силу» сглаживания.

### Stochastic Weight Averaging

Основная идея SWA связана с ландшафтом функции потерь нейронной сети. Представьте, что функция потерь — это горный рельеф, где мы ищем самую глубокую долину (глобальный минимум). Традиционное обучение с помощью стохастического градиентного спуска (SGD) похоже на спуск с горы в тумане: мы делаем шаги в направлении спуска, но можем застрять в локальном минимуме. SWA предлагает другой подход: вместо того чтобы использовать веса модели из последней эпохи обучения, мы собираем веса из разных точек траектории обучения и усредняем их. Пошагово алгоритм выполняет следующие действия:
1. Сначала модель обучается обычным способом (например, с помощью SGD) до момента, когда функция потерь начинает колебаться вокруг некоторого значения. Это означает, что мы достигли области с хорошими решениями.
2. После этого мы можем использовать циклический или постоянный Learning Rate. При циклическом подходе LR периодически меняется между заданными значениями, что позволяет модели исследовать разные области пространства решений.
3. На этом этапе мы периодически сохраняем веса модели. В конце обучения все собранные веса усредняются.

Более подробное объяснение можно найти, например, в блоге PyTorch — https://pytorch.org/blog/pytorch-1.6-now-includes-stochastic-weight-averaging/ — или в оригинальной статье — https://arxiv.org/abs/1803.05407. Примечание: метод можно использовать не только с SGD, но и с тем же Adam.

Документация PyTorch по работе с этими методами: https://pytorch.org/docs/stable/optim.html#weight-averaging-swa-and-ema. Предлагаем вам попробовать их в деле.

Внесите нужные изменения в функции для обучения модели, ориентируйтесь на документацию PyTorch:

In [ ]:
def run_epoch(model: nn.Module, loader: DataLoader, criterion: Callable, optimizer: Optional[torch.optim.Optimizer] = None,\
              scheduler: Optional[torch.optim.lr_scheduler.LRScheduler] = None, ens_method: Optional[str] = None,\
              ens_model: Optional[torch.optim.swa_utils.AveragedModel] = None, ens_start: bool = False, device: torch.device = torch.device("cpu")) -> torch.Tensor:
    all_labels, all_preds = [], []
    loss_epoch = 0.
    for batch in loader:
        images, labels = batch
        images, labels = images.to(device), labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        if optimizer is not None:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            ...

            if scheduler is not None and (ens_method != 'swa' or not ens_start):
                scheduler.step()

        loss_epoch += loss.item()
        preds = torch.argmax(logits.softmax(dim=-1), dim=-1)

        all_preds = np.concatenate((all_preds, preds.cpu().numpy()))
        all_labels = np.concatenate((all_labels, labels.cpu().numpy()))

    loss_epoch /= len(loader)
    acc_epoch = (all_preds == all_labels).sum() / len(all_preds)

    return loss_epoch, acc_epoch

def train(model: nn.Module, n_epochs: int, train_loader: DataLoader, criterion: Callable, optimizer: torch.optim.Optimizer,
          scheduler: Optional[torch.optim.lr_scheduler.LRScheduler] = None, val_loader: Optional[DataLoader] = None, val_freq: int = 10, ens_method: Optional[str] = None,\
          ens_model: Optional[torch.optim.swa_utils.AveragedModel] = None, ens_scheduler: Optional[torch.optim.swa_utils.SWALR] = None, ens_start_epoch: Optional[int] = None,\
          save_best: bool = True, save_name: str = 'model', device: torch.device = torch.device("cpu")) -> nn.Module:
    enable_validation = val_loader is not None
    best_val = 0.

    for epoch in range(n_epochs):
        timer_start = time()

        model.train()

        ens_start = (epoch > ens_start_epoch) if ens_start_epoch is not None else False
        train_loss_epoch, train_acc_epoch = run_epoch(model, train_loader, criterion, optimizer, scheduler, device)

        ...

        print(f"Epoch {epoch+1}:")
        print(f"Train loss: {train_loss_epoch} | Train acc: {train_acc_epoch * 100}%")

        if enable_validation and epoch % val_freq == 0:
            model.eval()
            with torch.no_grad():
                val_loss_epoch, val_acc_epoch = run_epoch(model, val_loader, criterion, optimizer=None, scheduler=None, device=device)

            if save_best and val_acc_epoch >= best_val:
                best_val = val_acc_epoch
                model.to("cpu")
                torch.save(model.state_dict(), f"{save_name}.pth")
                model.to(device)

            print(f"Val loss: {val_loss_epoch} | Val acc: {val_acc_epoch * 100}%")

        print(f"Time spent on epoch: {time() - timer_start}")

    return model

Возьмите пайплайн обучения из части 5 и добавьте EMA/SWA. Примечание: `ens_method` будет либо 'ema', либо 'swa'.

In [ ]:
train_transform = transforms.Compose([
    ...
])

val_transform = transforms.Compose([
    transforms.PILToTensor(),
    transforms.ToDtype(dtype=torch.float32, scale=True)
])

train_dataset, val_dataset = TinyImageNetDataset(train_df, train_transform), TinyImageNetDataset(val_df, val_transform)

train_loader, val_loader = DataLoader(train_dataset, batch_size=200, shuffle=True, pin_memory=True, drop_last=False),\
                           DataLoader(val_dataset, batch_size=200, pin_memory=True, shuffle=False)

n_epochs = ...

criterion = ...

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = MobileNetV3(3, 200)
model.to(device)

optimizer = ...
lr_scheduler = ...

...

model = train(model, n_epochs, train_loader, criterion, optimizer, lr_scheduler, val_loader, val_freq=1, save_name='model_p4', device=device)

...

Загрузите код модели и веса Averaged модели в LMS. Веса сохраняйте так же, как и с обычной моделью: `torch.save(ens_model.state_dict(), ...)`.

## Заключение

Итак, в этом домашнем задании мы рассмотрели основные подходы к построению продвинутого пайплайна для задачи классификации. В случае успешного выполнения всех заданий вы обнаружите, что только за счёт тюнинга процесса обучения можно улучшить метрику качества почти в 2 раза. Стоит отметить, что в реальных задачах вы, скорее всего, будете строить именно такие «продвинутые» пайплайны, где каждую компоненту тренировочного процесса нужно будет подтюнить для достижения наилучшего результата. За рамками данного ДЗ остались подходы вроде transfer learning, дистилляции и т. д., часть из них будет разобрана позднее, а пока можете почитать про эти техники самостоятельно.